(app:cookbook)=
# The Python cookbook

:::{only} html
```{button-link} cookbook.pdf
:color: primary
:outline:

Download the cookbook (PDF)
```
:::

This Python cookbook contains various routines or 'recipes' for the analysis of non-linear and chaotic dynamical systems. The aim of this document is to provide you with a set of recipes that will enable you to analyze physical systems quickly and effectively. By no means is this document intended to be a sufficient introduction into Python, although it should give a reasonable impression of how Python works and what it can do in this field.

## Getting started

The recipes in this cookbook are written for the Jupyter notebook, the
standard interactive environment for scientific Python. The easiest way to
get everything at once is to install [Anaconda](https://www.anaconda.com);
alternatively install the packages directly with

    pip install numpy scipy matplotlib sympy jupyterlab

and start the environment with `jupyter lab`.

A notebook consists of *cells*. A cell contains either code or text
(Markdown), and a code cell is executed with Shift-Enter. All code in this
cookbook can be typed into a code cell and run directly.

## Useful keys

- Shift-Enter: run cell, move to the next
- Ctrl-Enter: run cell, stay
- Alt-Enter: run cell, insert a new one below
- Esc / Enter: leave / enter a cell (command vs edit mode)
- A / B (in command mode): insert cell above / below
- D,D (in command mode): delete cell
- M / Y (in command mode): make cell Markdown / code
- Z (in command mode): undo cell operation

Try them now!

## Help

Help on any function is available with `help`, or — in the notebook — by
appending a question mark (`sum?`) or pressing Shift-Tab inside the
parentheses.

In [ ]:
help(sum)

## Comments

In [ ]:
#
# the #-sign allows comments
#

## General

A session normally starts with importing the packages we need: `numpy` for
numerical arrays, `matplotlib` for figures, `sympy` for symbolic
mathematics and `solve_ivp` from `scipy` for differential equations. The
short names `np`, `plt` and `sym` are conventional.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sym
from scipy.integrate import solve_ivp

Some important details:

- A cell displays the value of its **last** expression automatically.
- Ending a line with `;` suppresses that display.
- Assigning a value to a variable is done with `=`. A common mistake is to
  confuse it with `==`, which *compares* two values.

In [ ]:
a = 1 / np.sqrt(2)   # assignment: a gets a value
a

In [ ]:
a == 0.5             # comparison: is a equal to 0.5?

## Floating point numbers and exact answers

Python computes with floating point numbers, accurate to about 16
significant digits: `1/2` and `0.5` are the same thing. If an exact or
arbitrarily accurate answer is needed, `sympy` provides it: `sym.sqrt(2)`
stays exact, and `sym.N` evaluates to any number of digits.

In [ ]:
1 / 2

In [ ]:
a = 1 / sym.sqrt(2)
a

In [ ]:
sym.N(a)

In [ ]:
sym.N(sym.pi, 60)

## Functions vs expressions

A Python *function* computes numbers from numbers:

In [ ]:
def f(x):
    return x**2

f(2)

A sympy *expression* is a formula containing symbols. It is not evaluated
until values are substituted with `subs`:

In [ ]:
x, y = sym.symbols("x y")
q = x**2
q

In [ ]:
q.subs(x, 2)

The command `sym.diff` differentiates an expression:

In [ ]:
q = sym.sin(x**2)
dq = sym.diff(q, x)
dq

`sym.lambdify` converts an expression into an ordinary numerical function —
useful whenever a symbolic result (a derivative, say) is needed inside a
numerical recipe:

In [ ]:
dg = sym.lambdify(x, dq)
dg(2.0)

Expressions of more variables work the same way, and partial derivatives
are taken by naming the variable:

In [ ]:
r = sym.sqrt(x**2 + y**2)
sym.diff(r, x)

We prefer plain Python functions in the recipes that follow, and bring in
sympy expressions only where symbolic work (fixed points, derivatives,
stability) is the point.

## Plotting

Plotting a function means evaluating it on a fine grid of points and
drawing lines through the results. `np.linspace` creates the grid;
functions from `numpy` (`np.sin`, `np.sqrt`, ...) work on whole arrays at
once.

In [ ]:
xs = np.linspace(-2, 2, 200)
plt.plot(xs, np.sin(xs**2), label="sin(x²)")
plt.plot(xs, 2 * xs**2, label="2x²")
plt.xlabel("x")
plt.ylabel("y")
plt.legend();

## Lists, tuples and arrays

Python's basic container is the list, written with square brackets. Lists
preserve order and may contain doubles. **Indices start at 0.**

In [ ]:
a = [1, 2, 3, 2, 1]
a[2]

Appending elements one at a time — which we will do frequently to collect
results in a loop — starts from an empty list:

In [ ]:
c = []
c.append(1)
c.append(2)
c

A set, written with curly braces, discards doubles and order. `len` gives
the number of elements of any container.

In [ ]:
set(a)

In [ ]:
len(a)

A *list comprehension* generates a new list from a rule — we will need it
regularly:

In [ ]:
[i**2 for i in range(1, 6)]

For numerical work the `numpy` array is the workhorse. Arrays can be
created filled with zeros or ones, from a list, or from a rule; arithmetic
acts on all elements at once.

In [ ]:
np.zeros(10)

In [ ]:
np.ones(5)

In [ ]:
t = np.array([5, 3, 4, 1, 2])
t

In [ ]:
t = np.arange(1, 6)      # the integers 1..5
t**2

Sums and products of the elements:

In [ ]:
t2 = t**2
t2.sum(), t.prod()

Two-dimensional arrays are created the same way, with a pair of sizes:

In [ ]:
np.zeros((5, 5))

In [ ]:
i, j = np.indices((5, 5))
(i + 1)**2 + (j + 1)**2

Matrices and vectors are simply 2D and 1D arrays. `@` is the
matrix-vector product, and `np.linalg` contains the linear algebra
routines. Say we want to solve $A \mathbf{y} = \mathbf{b}$:

In [ ]:
A = np.array([[1, 2],
              [3, 4]])
b = np.array([1, 2])
ysol = np.linalg.solve(A, b)
ysol

Which is indeed the solution, as verified below.

In [ ]:
b - A @ ysol

(The inverse `np.linalg.inv(A)` exists too, but for solving a system
`np.linalg.solve` is the better tool.)

## For-loops

Frequently we will want to study the behavior of the equations over
complete parameter spaces, which can be done with for-loops. Here we
tabulate a sine function; `range(N)` runs `i` through `0, 1, ..., N-1`.

In [ ]:
N = 25
X = np.zeros(N)
Y = np.zeros(N)
for i in range(N):
    X[i] = i * 2 * np.pi / (N - 1)
    Y[i] = np.sin(X[i])
X[:10]

The slice `X[:10]` shows the first ten elements. The same table can be made
without a loop (`X = np.linspace(0, 2*np.pi, N); Y = np.sin(X)`), which is
faster — but the loop form generalizes to the iterative recipes below,
where each value depends on the previous one and there is no way around a
loop.

## Plotting data points

To plot discrete data points rather than a line, give `plt.plot` a marker
style: `"o"` for circles, `"s"` for squares, `"d"` for diamonds, `","` for
single pixels.

In [ ]:
plt.plot(X, Y, "o")
plt.xlabel("x")
plt.ylabel("y");

Lines and points combine naturally — a format string like `"r-"` (red
line) or `"ko"` (black circles) sets both color and style:

In [ ]:
pts = np.arange(0, 5)
xfine = np.linspace(0, 4, 100)
plt.plot(xfine, xfine**2, "r-")
plt.plot(pts, pts**2, "ko")
plt.xlabel("x")
plt.ylabel("y");

## Plotting more than one function at the same time

Successive `plt.plot` calls draw into the same figure until the cell ends,
so a loop can overlay any number of curves. `label` and `plt.legend` handle
the legend.

In [ ]:
def f(x, a):
    return np.sin(a * x)

xs = np.linspace(-np.pi, np.pi, 200)
for a in [1, 2, 3]:
    plt.plot(xs, f(xs, a), label=f"a={a}")
plt.xlabel("x")
plt.ylabel("y")
plt.legend();

## Implicit plots

A very useful trick is plotting *implicitly* defined data: the set of
points where some function is zero. `plt.contour` with the single contour
level 0 does exactly this — for example those $x$ for which
$x=\tanh(x/T)$, as a function of $T$:

In [ ]:
T, x = np.meshgrid(np.linspace(0.01, 2, 400), np.linspace(-1.1, 1.1, 400))
F = -x + np.tanh(x / T)
plt.contour(T, x, F, levels=[0], colors="magenta")
plt.xlabel("T")
plt.ylabel("x");

## Iterative maps

The logistic map $x_{n+1} = r\,x_n(1-x_n)$ iterated with a plain loop.
Each value depends on the previous one, so a loop is the natural tool.

In [ ]:
def f(x):
    return r * x * (1 - x)   # define function

N = 50                       # define nr of points
X = np.zeros(N + 1)          # declare data array
r = 0.5                      # set parameter
X[0] = 0.1                   # set initial condition
for n in range(N):           # perform iterations
    X[n + 1] = f(X[n])

Plot the time series as points:

In [ ]:
plt.plot(range(N + 1), X, "d")
plt.xlabel("n")
plt.ylabel("x[n]");

or as a line through circles:

In [ ]:
plt.plot(range(N + 1), X, "-o")
plt.xlabel("n")
plt.ylabel("x[n]");

## Generate cobweb

Create and plot a cobweb of the discrete data set: starting from
$(x_0, 0)$, repeatedly step vertically to the map and horizontally to the
diagonal.

In [ ]:
px = [X[0]]
py = [0]
for n in range(N):
    px += [X[n], X[n]]
    py += [X[n], X[n + 1]]

xs = np.linspace(0, 1, 200)
plt.plot(xs, f(xs), "k")     # the map
plt.plot(xs, xs, "b")        # the diagonal
plt.plot(px, py, "r")        # the cobweb
plt.axis([0, 1, 0, 1])
plt.xlabel("x[n]")
plt.ylabel("x[n+1]");

## Return plot

First we need to create a data series, now in the chaotic regime:

In [ ]:
def f(x):
    return r * x * (1 - x)

N = 200
r = 3.8
X = np.zeros(N + 1)
X[0] = 0.4
for n in range(N):
    X[n + 1] = f(X[n])

Now create a return plot from the $[x_n, x_{n+1}]$ pairs, removing the
transient by disregarding the first $N/2$ points:

In [ ]:
xs = np.linspace(0, 1, 200)
plt.plot(X[N // 2 : N], X[N // 2 + 1 : N + 1], "ro")
plt.plot(xs, f(xs), "k")
plt.axis([0, 1, 0, 1])
plt.xlabel("x[n]")
plt.ylabel("x[n+1]");

## Bifurcation diagram

For each value of $r$, iterate the map and keep only the second half of
the series (the part that has converged onto the attractor); plotting
those points against $r$ produces the bifurcation diagram. The `","`
marker plots single pixels.

In [ ]:
def f(x):
    return r * x * (1 - x)

N = 500                      # iterations per r value
Nr = 200                     # number of r values
rmin, rmax = 0.1, 4
Nmin = N // 2                # discard the first half as transient

rs = []
xs = []
for i in range(Nr + 1):
    r = rmin + (rmax - rmin) * i / Nr
    X = np.zeros(N + 1)
    X[0] = 0.57
    for n in range(N):
        X[n + 1] = f(X[n])
    for n in range(Nmin, N + 1):
        rs.append(r)
        xs.append(X[n])

plt.plot(rs, xs, ",", color="blue")
plt.axis([rmin, rmax, 0, 1])
plt.xlabel("r")
plt.ylabel("x(r)");